# Đánh giá Toàn diện Tất cả Mô hình Image Harmonization trong `runs/`

Notebook này chạy **cùng một pipeline đánh giá công bằng và thống nhất** trên toàn bộ **7.404 ảnh Test chính thức của iHarmony4 (256×256)** cho tất cả các mô hình có trong thư mục `ImageHarmonization/runs/`:

0. **Input Composite (No Harmonization)** — Baseline ảnh ghép gốc chưa xử lý
1. **Baseline U-Net (Full L1)** — `unet_full_l1_all4_bs32_seed42/U-NET_Full_L1_checkpoint.pt`
2. **Baseline U-Net (Masked L1)** — `unet_masked_l1_50ep_all4_bs32_seed42/best.pt`
3. **U-Net + 1 PatchGAN (Branch B)** — `unet_branchB_1patchgan_50ep_all4_bs32_seed42/best.pt`
4. **U-Net + 3 Multi-scale PatchGAN (Branch C)** — `unet_branchC_3patchgan_50ep_all4_bs32_seed42/best.pt`
5. **Improved U-Net v2 (Hybrid)** — `improved_unet_v2_hybrid_probe10ep_bs32_seed42/best.pt`
6. **Improved U-Net v3 (DINOv2 + Lab)** — `improved_unet_v3_dinov2_lab_30ep_bs32_seed42/best.pt`
7. **RainNet Pretrained (CVPR 2021)** — `RainNet_Pretrained/RainNet_net_G_last.pth`
8. **HT+ Pretrained (`HT_2H9L_allihd`)** — `ht_plus_pretrained/HT_2H9L_allihd_latest_net_G.pth`

### Nội dung đánh giá
- **7 chỉ số định lượng đồng nhất** (float32, ghép nền gốc `out = pred * m + comp * (1 - m)`): `MSE ↓`, `fMSE ↓`, `PSNR ↑`, `fPSNR ↑`, `SSIM ↑`, `fSSIM ↑`, `fDeltaE76 ↓` (CIE Lab D65) + **tốc độ suy luận (ms/ảnh, FPS)** và **số tham số**.
- **Xác minh sạch dữ liệu:** 0 rò rỉ Train/Val → Test và kiểm tra khớp mã băm SHA-256 `fingerprint` cho tất cả checkpoint tự huấn luyện.
- **Phân tích đa chiều:** Tổng hợp toàn bộ 7.404 ảnh, chia theo **4 sub-dataset** (`HCOCO`, `HAdobe5k`, `HFlickr`, `Hday2night`), và chia theo **6 nhóm độ khó** từ `iHarmony4_groups_train_test/test.csv` (`small/large` × `low/medium/high`).
- **Xuất báo cáo tự động:** Lưu toàn bộ bảng `.csv`, `.json`, điểm từng ảnh (`per_image_metrics/*.csv`), lưới ảnh trực quan và biểu đồ vào `runs/evaluation_all_models/`.
- **Phần 7 — Kiểm định riêng cho RainNet (Không gian lận vs Đúng cách đo repo gốc `junleen/RainNet`):** Dùng đúng checkpoint `RainNet_Pretrained/RainNet_net_G_last.pth` (256×256) đang có trong `runs/` để đối chiếu trực tiếp 4 cách đo: **Blind không gian lận** (`comp` gốc) vs **GT-BG Input + Fair Comp BG** vs **Protocol gốc `junleen/RainNet` (Float32 & `evaluate.py` uint8)**.


In [1]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

import torch
assert torch.cuda.is_available(), 'Hãy bật GPU (T4 / L4 / A100) trong Runtime -> Change runtime type trước khi chạy!'
DEVICE = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU sẵn sàng: {gpu_name} ({vram_gb:.1f} GB VRAM) | PyTorch {torch.__version__}')

# ==============================================================================
# ĐỊNH VỊ THƯ MỤC DỰ ÁN TRÊN GOOGLE DRIVE
# ==============================================================================
PROJECT_ROOT = Path('/content/drive/MyDrive/ImageHarmonization')
ZIP_DIR = PROJECT_ROOT / 'iHarmony4_dataset_resize_256'
GROUPS_DIR = PROJECT_ROOT / 'iHarmony4_groups_train_test'

# Ưu tiên đúng cấu trúc chuẩn ImageHarmonization/runs (fallback sang thư mục cũ nếu cần)
CANDIDATE_RUNS_DIRS = [
    PROJECT_ROOT / 'runs',
    ZIP_DIR / 'runs',
]
RUNS_DIR = next((d for d in CANDIDATE_RUNS_DIRS if d.is_dir()), PROJECT_ROOT / 'runs')
OUT_DIR = RUNS_DIR / 'evaluation_all_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'per_image_metrics').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'previews').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'charts').mkdir(parents=True, exist_ok=True)

def resolve_run_file(rel_path: str) -> Path:
    """Tìm checkpoint trong ImageHarmonization/runs trước, fallback sang iHarmony4_dataset_resize_256/runs."""
    for base in CANDIDATE_RUNS_DIRS:
        p = base / rel_path
        if p.is_file():
            return p
    return RUNS_DIR / rel_path

DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']

for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = ZIP_DIR / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RUNS_DIR    :', RUNS_DIR)
print('OUT_DIR     :', OUT_DIR)
print('DATA 256x256:', DATA)


Mounted at /content/drive
GPU sẵn sàng: Tesla T4 (14.6 GB VRAM) | PyTorch 2.14.1+cu130
Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
PROJECT_ROOT: /content/drive/MyDrive/ImageHarmonization
RUNS_DIR    : /content/drive/MyDrive/ImageHarmonization/runs
OUT_DIR     : /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models
DATA 256x256: /content/iharmony4_256


## Phần 2: Định nghĩa đầy đủ 5 kiến trúc mạng (tự chứa 100%)
Bao gồm:
1. `UNet` (Baseline Full L1, Masked L1, Branch B 1-PatchGAN, Branch C 3-PatchGAN)
2. `ImprovedHarmonizationUNet` (v2 Hybrid: GatedConv + Bottleneck FG-BG Cross-Attention + Attention Gates + Color Affine Head)
3. `ImprovedHarmonizationUNetV3` (v3 DINOv2 + Lab: Frozen DINOv2 ViT-S/14 Semantic Guide + Hybrid Affine Residual Head)
4. `RainNet` (CVPR 2021 — Region-aware Adaptive Instance Normalization `RAIN`, không cần clone repo hay cài `yacs`)
5. `HTGenerator` (`HT_2H9L_allihd` — Harmony Transformer 2-head 9-layer với Sine Positional Embedding, không cần `einops`/`visdom`)

In [2]:
import copy
import functools
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# ==============================================================================
# 1. CÁC BLOCK DÙNG CHUNG CHO IMPROVED U-NET v2 & v3
# ==============================================================================
class GatedConv2d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding, bias=False)
        self.norm = nn.GroupNorm(num_groups=math.gcd(8, out_ch), num_channels=out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        feat, gate = self.conv(x).chunk(2, dim=1)
        return self.act(self.norm(feat)) * torch.sigmoid(gate)

class GatedDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(GatedConv2d(in_ch, out_ch), GatedConv2d(out_ch, out_ch))

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

class StandardDoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        num_groups = math.gcd(8, out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

class BilinearUp(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.up(x)

class AttentionGate(nn.Module):
    def __init__(self, g_ch: int, x_ch: int):
        super().__init__()
        inter_ch = max(x_ch // 2, 16)
        self.W_g = nn.Conv2d(g_ch, inter_ch, kernel_size=1, bias=True)
        self.W_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(inter_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, g: Tensor, x: Tensor) -> Tensor:
        return x * self.psi(self.W_g(g) + self.W_x(x))


# ==============================================================================
# 2. KIẾN TRÚC A: BASELINE U-NET (Full L1, Masked L1, Branch B, Branch C)
# ==============================================================================
class Block(nn.Sequential):
    def __init__(self, cin: int, cout: int):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, b: int = 32):
        super().__init__()
        self.enc = nn.ModuleList([Block(4, b), Block(b, b*2), Block(b*2, b*4), Block(b*4, b*8), Block(b*8, b*16)])
        self.up = nn.ModuleList([
            nn.ConvTranspose2d(b*16, b*8, 2, 2), nn.ConvTranspose2d(b*8, b*4, 2, 2),
            nn.ConvTranspose2d(b*4, b*2, 2, 2), nn.ConvTranspose2d(b*2, b, 2, 2),
        ])
        self.dec = nn.ModuleList([Block(b*16, b*8), Block(b*8, b*4), Block(b*4, b*2), Block(b*2, b)])
        self.head = nn.Conv2d(b, 3, 1)

    def forward(self, x: Tensor) -> Tensor:
        skips = []
        for i, block in enumerate(self.enc):
            x = block(x)
            if i < 4:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, block, skip in zip(self.up, self.dec, reversed(skips)):
            x = block(torch.cat([up(x), skip], 1))
        return self.head(x).sigmoid()


# ==============================================================================
# 3. KIẾN TRÚC B: IMPROVED U-NET v2 (Hybrid)
# ==============================================================================
class BottleneckFGBGCrossAttention(nn.Module):
    def __init__(self, channels: int = 512, num_heads: int = 8, dropout: float = 0.0):
        super().__init__()
        self.norm_q = nn.LayerNorm(channels)
        self.norm_kv = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.gamma1 = nn.Parameter(torch.zeros(1))
        self.gamma2 = nn.Parameter(torch.zeros(1))

    def forward(self, feat: Tensor, mask: Tensor) -> Tensor:
        B, C, H, W = feat.shape
        m_low = F.adaptive_avg_pool2d(mask, (H, W))
        m_flat = m_low.view(B, H * W)
        tokens = feat.flatten(2).transpose(1, 2)
        q, kv = self.norm_q(tokens), self.norm_kv(tokens)

        key_padding_mask = m_flat > 0.5
        all_fg = key_padding_mask.all(dim=1)
        if all_fg.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[all_fg] = False

        attn_out, _ = self.attn(query=q, key=kv, value=kv, key_padding_mask=key_padding_mask, need_weights=False)
        fg_gate_tokens = (m_flat > 0.2).unsqueeze(-1).to(feat.dtype)
        tokens = tokens + self.gamma1 * (attn_out * fg_gate_tokens)
        tokens = tokens + self.gamma2 * (self.ffn(self.norm_ffn(tokens)) * fg_gate_tokens)
        return tokens.transpose(1, 2).view(B, C, H, W)

class GlobalLocalColorAffineHead(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.local_conv = nn.Conv2d(in_ch, 12, kernel_size=3, padding=1)
        self.global_fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, 12))

    def forward(self, feat: Tensor, comp: Tensor) -> Tensor:
        B, _, H, W = feat.shape
        params = self.local_conv(feat) + self.global_fc(feat).view(B, 12, 1, 1)
        dW = params[:, :9].view(B, 3, 3, H, W)
        db = params[:, 9:].view(B, 3, H, W)
        eye = torch.eye(3, device=feat.device, dtype=feat.dtype).view(1, 3, 3, 1, 1)
        pred = torch.einsum("bijhw,bjhw->bihw", (eye + dW).float(), comp.float()) + db.float()
        return torch.clamp(pred, 0.0, 1.0).to(feat.dtype)

class ImprovedHarmonizationUNet(nn.Module):
    def __init__(self, base_channels: int = 32, num_heads: int = 8):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels*2, base_channels*4, base_channels*8, base_channels*16
        self.enc1 = GatedDoubleConv(4, c1)
        self.enc2 = GatedDoubleConv(c1, c2)
        self.enc3 = GatedDoubleConv(c2, c3)
        self.enc4 = GatedDoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = GatedDoubleConv(c4, c5)
        self.fg_bg_attn = BottleneckFGBGCrossAttention(channels=c5, num_heads=num_heads)

        self.up4 = BilinearUp(c5, c4)
        self.att4 = AttentionGate(g_ch=c4, x_ch=c4)
        self.dec4 = StandardDoubleConv(c4 * 2, c4)

        self.up3 = BilinearUp(c4, c3)
        self.att3 = AttentionGate(g_ch=c3, x_ch=c3)
        self.dec3 = StandardDoubleConv(c3 * 2, c3)

        self.up2 = BilinearUp(c3, c2)
        self.att2 = AttentionGate(g_ch=c2, x_ch=c2)
        self.dec2 = StandardDoubleConv(c2 * 2, c2)

        self.up1 = BilinearUp(c2, c1)
        self.att1 = AttentionGate(g_ch=c1, x_ch=c1)
        self.dec1 = StandardDoubleConv(c1 * 2, c1)

        self.head = GlobalLocalColorAffineHead(c1)

    def forward(self, x: Tensor) -> Tensor:
        comp, mask = x[:, :3], x[:, 3:4]
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        bot = self.fg_bg_attn(self.bottleneck(self.pool(s4)), mask)

        g4 = self.up4(bot)
        d4 = self.dec4(torch.cat([g4, self.att4(g4, s4)], dim=1))
        g3 = self.up3(d4)
        d3 = self.dec3(torch.cat([g3, self.att3(g3, s3)], dim=1))
        g2 = self.up2(d3)
        d2 = self.dec2(torch.cat([g2, self.att2(g2, s2)], dim=1))
        g1 = self.up1(d2)
        d1 = self.dec1(torch.cat([g1, self.att1(g1, s1)], dim=1))
        return self.head(d1, comp)


# ==============================================================================
# 4. KIẾN TRÚC C: IMPROVED U-NET v3 (DINOv2 + Lab)
# ==============================================================================
class FrozenDINOv2Guide(nn.Module):
    def __init__(self, out_channels: int = 512):
        super().__init__()
        self.dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
        for p in self.dino.parameters():
            p.requires_grad = False
        self.dino.eval()

        self.proj = nn.Sequential(
            nn.Conv2d(384, out_channels, kernel_size=1, bias=False),
            nn.GroupNorm(8, out_channels),
        )
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def train(self, mode: bool = True):
        super().train(mode)
        self.dino.eval()
        return self

    def forward(self, rgb: Tensor, target_hw: tuple[int, int]) -> Tensor:
        with torch.no_grad():
            x = F.interpolate(rgb, size=(224, 224), mode="bilinear", align_corners=False)
            x = (x - self.mean) / self.std
            feats = self.dino.forward_features(x)["x_norm_patchtokens"]
            B, N, C = feats.shape
            H_p = W_p = int(math.sqrt(N))
            feat_map = feats.transpose(1, 2).view(B, C, H_p, W_p)

        if (H_p, W_p) != target_hw:
            feat_map = F.interpolate(feat_map, size=target_hw, mode="bilinear", align_corners=False)
        return self.proj(feat_map)

class SemanticGuidedFGBGCrossAttention(nn.Module):
    def __init__(self, channels: int = 512, num_heads: int = 8):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(1, 256, channels) * 0.02)
        self.norm_sem_q = nn.LayerNorm(channels)
        self.norm_sem_k = nn.LayerNorm(channels)
        self.norm_val = nn.LayerNorm(channels)

        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.gamma1 = nn.Parameter(torch.zeros(1))
        self.gamma2 = nn.Parameter(torch.zeros(1))

    def forward(self, spatial_feat: Tensor, sem_feat: Tensor, mask: Tensor) -> Tensor:
        B, C, H, W = spatial_feat.shape
        m_low = F.adaptive_avg_pool2d(mask, (H, W))
        m_flat = m_low.view(B, H * W)

        sp_tokens = spatial_feat.flatten(2).transpose(1, 2)
        sem_tokens = sem_feat.flatten(2).transpose(1, 2)

        pos = self.pos_emb if H * W == self.pos_emb.shape[1] else F.interpolate(
            self.pos_emb.transpose(1, 2).view(1, C, 16, 16), size=(H, W), mode="bilinear", align_corners=False
        ).flatten(2).transpose(1, 2)

        q = self.norm_sem_q(sem_tokens + pos)
        k = self.norm_sem_k(sem_tokens + pos)
        v = self.norm_val(sp_tokens)

        bg_key_padding_mask = m_flat > 0.5
        all_fg = bg_key_padding_mask.all(dim=1)
        if all_fg.any():
            bg_key_padding_mask = bg_key_padding_mask.clone()
            bg_key_padding_mask[all_fg] = False

        attn_out, _ = self.attn(query=q, key=k, value=v, key_padding_mask=bg_key_padding_mask, need_weights=False)

        fg_gate = (m_flat > 0.2).unsqueeze(-1).to(spatial_feat.dtype)
        sp_tokens = sp_tokens + self.gamma1 * (attn_out * fg_gate)
        sp_tokens = sp_tokens + self.gamma2 * (self.ffn(self.norm_ffn(sp_tokens)) * fg_gate)
        return sp_tokens.transpose(1, 2).view(B, C, H, W)

class HybridColorAffineResidualHead(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.local_conv = nn.Conv2d(in_ch, 12, kernel_size=3, padding=1)
        self.global_fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, 12))

        self.chroma_res = nn.Sequential(
            nn.Conv2d(in_ch + 3, in_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, in_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(in_ch, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )
        self.res_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, feat: Tensor, comp: Tensor, mask: Tensor) -> Tensor:
        B, _, H, W = feat.shape
        params = self.local_conv(feat) + self.global_fc(feat).view(B, 12, 1, 1)
        dW = params[:, :9].view(B, 3, 3, H, W)
        db = params[:, 9:].view(B, 3, H, W)
        eye = torch.eye(3, device=feat.device, dtype=feat.dtype).view(1, 3, 3, 1, 1)

        affine_pred = torch.einsum("bijhw,bjhw->bihw", (eye + dW).float(), comp.float()) + db.float()
        delta_rgb = self.chroma_res(torch.cat([feat, affine_pred.clamp(0.0, 1.0).to(feat.dtype)], dim=1)).float()

        pred = affine_pred + self.res_scale.float() * delta_rgb * mask.float()
        return torch.clamp(pred, 0.0, 1.0).to(feat.dtype)

class ImprovedHarmonizationUNetV3(nn.Module):
    def __init__(self, base_channels: int = 32, num_heads: int = 8):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels*2, base_channels*4, base_channels*8, base_channels*16

        self.enc1 = GatedDoubleConv(4, c1)
        self.enc2 = GatedDoubleConv(c1, c2)
        self.enc3 = StandardDoubleConv(c2, c3)
        self.enc4 = StandardDoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = StandardDoubleConv(c4, c5)
        self.sem_guide = FrozenDINOv2Guide(out_channels=c5)
        self.fg_bg_attn = SemanticGuidedFGBGCrossAttention(channels=c5, num_heads=num_heads)

        self.up4 = BilinearUp(c5, c4)
        self.att4 = AttentionGate(g_ch=c4, x_ch=c4)
        self.dec4 = StandardDoubleConv(c4 * 2, c4)

        self.up3 = BilinearUp(c4, c3)
        self.att3 = AttentionGate(g_ch=c3, x_ch=c3)
        self.dec3 = StandardDoubleConv(c3 * 2, c3)

        self.up2 = BilinearUp(c3, c2)
        self.att2 = AttentionGate(g_ch=c2, x_ch=c2)
        self.dec2 = StandardDoubleConv(c2 * 2, c2)

        self.up1 = BilinearUp(c2, c1)
        self.att1 = AttentionGate(g_ch=c1, x_ch=c1)
        self.dec1 = StandardDoubleConv(c1 * 2, c1)

        self.head = HybridColorAffineResidualHead(c1)

    def forward(self, x: Tensor) -> Tensor:
        comp, mask = x[:, :3], x[:, 3:4]
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))

        bot_spatial = self.bottleneck(self.pool(s4))
        bot_sem = self.sem_guide(comp, target_hw=bot_spatial.shape[-2:])
        bot = self.fg_bg_attn(bot_spatial, bot_sem, mask)

        g4 = self.up4(bot)
        d4 = self.dec4(torch.cat([g4, self.att4(g4, s4)], dim=1))
        g3 = self.up3(d4)
        d3 = self.dec3(torch.cat([g3, self.att3(g3, s3)], dim=1))
        g2 = self.up2(d3)
        d2 = self.dec2(torch.cat([g2, self.att2(g2, s2)], dim=1))
        g1 = self.up1(d2)
        d1 = self.dec1(torch.cat([g1, self.att1(g1, s1)], dim=1))
        return self.head(d1, comp, mask)


# ==============================================================================
# 5. KIẾN TRÚC D: RAINNET PRETRAINED (Ling et al., CVPR 2021)
# Khớp 100% state_dict của junleen/RainNet (RainNet_net_G_last.pth, 54.76M params)
# ==============================================================================
class RAIN(nn.Module):
    def __init__(self, dims_in: int, eps: float = 1e-5):
        super().__init__()
        self.foreground_gamma = nn.Parameter(torch.zeros(dims_in), requires_grad=True)
        self.foreground_beta = nn.Parameter(torch.zeros(dims_in), requires_grad=True)
        self.background_gamma = nn.Parameter(torch.zeros(dims_in), requires_grad=True)
        self.background_beta = nn.Parameter(torch.zeros(dims_in), requires_grad=True)
        self.eps = eps

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        mask = F.interpolate(mask.detach(), size=x.size()[2:], mode='nearest')
        mean_back, std_back = self.get_foreground_mean_std(x * (1 - mask), 1 - mask)
        normalized = (x - mean_back) / std_back
        normalized_background = (
            normalized * (1 + self.background_gamma[None, :, None, None])
            + self.background_beta[None, :, None, None]
        ) * (1 - mask)

        mean_fore, std_fore = self.get_foreground_mean_std(x * mask, mask)
        normalized = (x - mean_fore) / std_fore * std_back + mean_back
        normalized_foreground = (
            normalized * (1 + self.foreground_gamma[None, :, None, None])
            + self.foreground_beta[None, :, None, None]
        ) * mask
        return normalized_foreground + normalized_background

    def get_foreground_mean_std(self, region: Tensor, mask: Tensor):
        s = torch.sum(region, dim=[2, 3])
        num = torch.sum(mask, dim=[2, 3])
        mu = s / (num + self.eps)
        mean = mu[:, :, None, None]
        var = torch.sum((region + (1 - mask) * mean - mean) ** 2, dim=[2, 3]) / (num + self.eps)
        var = var[:, :, None, None]
        return mean, torch.sqrt(var + self.eps)

def _rain_inst_norm(ch: int):
    return nn.InstanceNorm2d(ch, affine=False, track_running_stats=False)

class UnetBlockCodec(nn.Module):
    def __init__(self, outer_nc, inner_nc, input_nc=None, submodule=None, outermost=False,
                 innermost=False, norm_layer=RAIN, use_dropout=False, use_attention=False, enc=True, dec=True):
        super().__init__()
        self.outermost = outermost
        self.innermost = innermost
        self.use_dropout = use_dropout
        self.use_attention = use_attention
        if input_nc is None:
            input_nc = outer_nc

        if outermost:
            self.down = nn.Conv2d(input_nc, inner_nc, kernel_size=4, stride=2, padding=1, bias=False)
            self.submodule = submodule
            self.up = nn.Sequential(
                nn.ReLU(True),
                nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1),
                nn.Tanh(),
            )
        elif innermost:
            self.up = nn.Sequential(
                nn.LeakyReLU(0.2, True),
                nn.Conv2d(input_nc, inner_nc, kernel_size=4, stride=2, padding=1, bias=False),
                nn.ReLU(True),
                nn.ConvTranspose2d(inner_nc, outer_nc, kernel_size=4, stride=2, padding=1, bias=False),
            )
            self.upnorm = norm_layer(outer_nc) if dec else _rain_inst_norm(outer_nc)
        else:
            self.down = nn.Sequential(
                nn.LeakyReLU(0.2, True),
                nn.Conv2d(input_nc, inner_nc, kernel_size=4, stride=2, padding=1, bias=False),
            )
            self.downnorm = norm_layer(inner_nc) if enc else _rain_inst_norm(inner_nc)
            self.submodule = submodule
            self.up = nn.Sequential(
                nn.ReLU(True),
                nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1, bias=False),
            )
            self.upnorm = norm_layer(outer_nc) if dec else _rain_inst_norm(outer_nc)
            if use_dropout:
                self.dropout = nn.Dropout(0.5)

        if use_attention:
            self.attention = nn.Sequential(
                nn.Conv2d(outer_nc + input_nc, outer_nc + input_nc, kernel_size=1),
                nn.Sigmoid(),
            )

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        if self.outermost:
            return self.up(self.submodule(self.down(x), mask))
        elif self.innermost:
            ret = self.up(x)
            ret = self.upnorm(ret, mask) if isinstance(self.upnorm, RAIN) else self.upnorm(ret)
            ret = torch.cat([x, ret], 1)
            return self.attention(ret) * ret if self.use_attention else ret
        else:
            ret = self.down(x)
            ret = self.downnorm(ret, mask) if isinstance(self.downnorm, RAIN) else self.downnorm(ret)
            ret = self.submodule(ret, mask)
            ret = self.up(ret)
            ret = self.upnorm(ret, mask) if isinstance(self.upnorm, RAIN) else self.upnorm(ret)
            if self.use_dropout:
                ret = self.dropout(ret)
            ret = torch.cat([x, ret], 1)
            return self.attention(ret) * ret if self.use_attention else ret

class RainNet(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, norm_layer=RAIN,
                 norm_type_indicator=(0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1),
                 use_dropout=True, use_attention=True):
        super().__init__()
        self.input_nc = input_nc
        self.use_attention = use_attention
        norm_type_list = [_rain_inst_norm, norm_layer]

        self.model_layer0 = nn.Conv2d(input_nc, ngf, kernel_size=4, stride=2, padding=1, bias=False)
        self.model_layer1 = nn.Sequential(nn.LeakyReLU(0.2, True), nn.Conv2d(ngf, ngf*2, 4, 2, 1, bias=False))
        self.model_layer1norm = norm_type_list[norm_type_indicator[0]](ngf * 2)

        self.model_layer2 = nn.Sequential(nn.LeakyReLU(0.2, True), nn.Conv2d(ngf*2, ngf*4, 4, 2, 1, bias=False))
        self.model_layer2norm = norm_type_list[norm_type_indicator[1]](ngf * 4)

        self.model_layer3 = nn.Sequential(nn.LeakyReLU(0.2, True), nn.Conv2d(ngf*4, ngf*8, 4, 2, 1, bias=False))
        self.model_layer3norm = norm_type_list[norm_type_indicator[2]](ngf * 8)

        unet_block = UnetBlockCodec(ngf*8, ngf*8, submodule=None, norm_layer=norm_layer,
                                    innermost=True, enc=norm_type_indicator[6], dec=norm_type_indicator[7])
        unet_block = UnetBlockCodec(ngf*8, ngf*8, submodule=unet_block, norm_layer=norm_layer,
                                    use_dropout=use_dropout, enc=norm_type_indicator[5], dec=norm_type_indicator[8])
        unet_block = UnetBlockCodec(ngf*8, ngf*8, submodule=unet_block, norm_layer=norm_layer,
                                    use_dropout=use_dropout, enc=norm_type_indicator[4], dec=norm_type_indicator[9])
        self.unet_block = UnetBlockCodec(ngf*8, ngf*8, submodule=unet_block, norm_layer=norm_layer,
                                         use_dropout=use_dropout, enc=norm_type_indicator[3], dec=norm_type_indicator[10])

        self.model_layer11 = nn.Sequential(nn.ReLU(True), nn.ConvTranspose2d(ngf*16, ngf*4, 4, 2, 1, bias=False))
        self.model_layer11norm = norm_type_list[norm_type_indicator[11]](ngf * 4)
        if use_attention:
            self.model_layer11att = nn.Sequential(nn.Conv2d(ngf*8, ngf*8, 1, 1), nn.Sigmoid())

        self.model_layer12 = nn.Sequential(nn.ReLU(True), nn.ConvTranspose2d(ngf*8, ngf*2, 4, 2, 1, bias=False))
        self.model_layer12norm = norm_type_list[norm_type_indicator[12]](ngf * 2)
        if use_attention:
            self.model_layer12att = nn.Sequential(nn.Conv2d(ngf*4, ngf*4, 1, 1), nn.Sigmoid())

        self.model_layer13 = nn.Sequential(nn.ReLU(True), nn.ConvTranspose2d(ngf*4, ngf, 4, 2, 1, bias=False))
        self.model_layer13norm = norm_type_list[norm_type_indicator[13]](ngf)
        if use_attention:
            self.model_layer13att = nn.Sequential(nn.Conv2d(ngf*2, ngf*2, 1, 1), nn.Sigmoid())

        self.model_out = nn.Sequential(nn.ReLU(True), nn.ConvTranspose2d(ngf*2, output_nc, 4, 2, 1), nn.Tanh())

    def _apply_norm(self, layer, x, mask):
        return layer(x, mask) if isinstance(layer, RAIN) else layer(x)

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        x0 = self.model_layer0(x)
        x1 = self._apply_norm(self.model_layer1norm, self.model_layer1(x0), mask)
        x2 = self._apply_norm(self.model_layer2norm, self.model_layer2(x1), mask)
        x3 = self._apply_norm(self.model_layer3norm, self.model_layer3(x2), mask)

        ox3 = self.unet_block(x3, mask)
        ox2 = torch.cat([x2, self._apply_norm(self.model_layer11norm, self.model_layer11(ox3), mask)], 1)
        if self.use_attention:
            ox2 = self.model_layer11att(ox2) * ox2

        ox1 = torch.cat([x1, self._apply_norm(self.model_layer12norm, self.model_layer12(ox2), mask)], 1)
        if self.use_attention:
            ox1 = self.model_layer12att(ox1) * ox1

        ox0 = torch.cat([x0, self._apply_norm(self.model_layer13norm, self.model_layer13(ox1), mask)], 1)
        if self.use_attention:
            ox0 = self.model_layer13att(ox0) * ox0

        return self.model_out(ox0)


# ==============================================================================
# 6. KIẾN TRÚC E: HARMONY TRANSFORMER (HT+ / HT_2H9L_allihd)
# Khớp 100% state_dict của zhenglab/HarmonyTransformer (4.77M params)
# ==============================================================================
class PatchRearrange(nn.Module):
    """Tương đương einops Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)')."""
    def __init__(self, p: int = 4):
        super().__init__()
        self.p = p

    def forward(self, x: Tensor) -> Tensor:
        B, C, H, W = x.shape
        p = self.p
        return x.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 3, 5, 1).reshape(B, (H // p) * (W // p), p * p * C)

class HTTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=256, nhead=2, dim_feedforward=512, dropout=0.0, activation="gelu"):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = F.gelu if activation == "gelu" else F.relu

    def forward(self, src: Tensor, pos: Optional[Tensor] = None) -> Tensor:
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, need_weights=False)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = self.norm2(src + self.dropout2(src2))
        return src

class HTTransformerEncoder(nn.Module):
    def __init__(self, encoder_layer: nn.Module, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer) for _ in range(num_layers)])

    def forward(self, src: Tensor, pos: Optional[Tensor] = None) -> Tensor:
        out = src
        for layer in self.layers:
            out = layer(out, pos=pos)
        return out

class HTTransformerEncoders(nn.Module):
    def __init__(self, d_model=256, nhead=2, num_encoder_layers=9, dim_feedforward=512, dropout=0.0, activation="gelu"):
        super().__init__()
        layer = HTTransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, activation)
        self.encoder = HTTransformerEncoder(layer, num_encoder_layers)

    def forward(self, src: Tensor, src_pos: Optional[Tensor] = None) -> Tensor:
        return self.encoder(src, pos=src_pos)

def _build_ht_sine_pos(size: int = 256, ksize: int = 4, stride: int = 4, dim: int = 256) -> Tensor:
    feature_h = (size - ksize) // stride + 1
    num_pos_feats = dim // 2
    ones = torch.ones((feature_h, feature_h), dtype=torch.float32)
    y_embed = ones.cumsum(0, dtype=torch.float32)
    x_embed = ones.cumsum(1, dtype=torch.float32)
    dim_t = torch.arange(num_pos_feats, dtype=torch.float32)
    dim_t = 10000.0 ** (2 * (dim_t // 2) / num_pos_feats)
    pos_x = x_embed[:, :, None] / dim_t
    pos_y = y_embed[:, :, None] / dim_t
    pos_x = torch.stack((pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3).flatten(2)
    pos_y = torch.stack((pos_y[:, :, 0::2].sin(), pos_y[:, :, 1::2].cos()), dim=3).flatten(2)
    pos = torch.cat((pos_y, pos_x), dim=2).permute(2, 0, 1)  # (dim, H_f, W_f)
    return pos.flatten(1).transpose(0, 1).unsqueeze(1).contiguous()  # (H_f*W_f, 1, dim)

class HTGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ksize=4, dim=256, nhead=2, num_layers=9, dim_forward=2, size=256):
        super().__init__()
        self.patch_to_embedding = nn.Sequential(
            PatchRearrange(p=ksize),
            nn.Linear(ksize * ksize * (input_nc + 1), dim),
        )
        self.transformer_enc = HTTransformerEncoders(
            d_model=dim, nhead=nhead, num_encoder_layers=num_layers,
            dim_feedforward=dim * dim_forward, dropout=0.0, activation="gelu",
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(dim, output_nc, kernel_size=ksize, stride=ksize, padding=0),
            nn.Tanh(),
        )
        # persistent=False để load_state_dict(strict=True) khớp 100% file .pth gốc
        self.register_buffer("base_pos", _build_ht_sine_pos(size=size, ksize=ksize, stride=ksize, dim=dim), persistent=False)

    def forward(self, inputs: Tensor) -> Tensor:
        patch_embedding = self.patch_to_embedding(inputs)  # (B, L, C)
        bs, L, C = patch_embedding.size()
        pixel_pos = self.base_pos.expand(-1, bs, -1)
        content = self.transformer_enc(patch_embedding.permute(1, 0, 2), src_pos=pixel_pos)
        s = int(math.sqrt(L))
        return self.dec(content.permute(1, 2, 0).contiguous().view(bs, C, s, s))


# ==============================================================================
# 7. FACTORY NẠP MÔ HÌNH & HÀM SUY LUẬN THỐNG NHẤT
# ==============================================================================
def strip_module_prefix(state_dict: dict) -> dict:
    if any(k.startswith("module.") for k in state_dict.keys()):
        return {k[len("module."):] if k.startswith("module.") else k: v for k, v in state_dict.items()}
    return state_dict

def load_harmonization_model(arch: str, ck_path, device: torch.device, expected_fp: Optional[str] = None):
    """Trả về (model_or_None, meta_dict). Hỗ trợ cả 6 checkpoint tự train và 2 checkpoint pretrained."""
    if arch == 'identity':
        return None, {'epoch': '-', 'trainable_params_M': 0.0, 'total_params_M': 0.0, 'val_fMSE': None}

    ck = torch.load(ck_path, map_location='cpu', weights_only=False)

    if arch in ('unet', 'improved_unet_v2', 'improved_unet_v3'):
        if expected_fp is not None and 'fingerprint' in ck:
            assert expected_fp == ck['fingerprint'], f"CẢNH BÁO: Split fingerprint không khớp cho {ck_path.name}!"
        cfg = ck.get('cfg', {})
        base_ch = cfg.get('base_channels', 32)
        num_heads = cfg.get('num_heads', 8)
        if arch == 'unet':
            model = UNet(base_ch)
        elif arch == 'improved_unet_v2':
            model = ImprovedHarmonizationUNet(base_channels=base_ch, num_heads=num_heads)
        else:
            model = ImprovedHarmonizationUNetV3(base_channels=base_ch, num_heads=num_heads)
        model.load_state_dict(strip_module_prefix(ck['model']), strict=True)
        val_fmse = ck.get('val', {}).get('all', {}).get('fMSE')
        meta = {
            'epoch': ck.get('epoch', '?'),
            'val_fMSE': round(float(val_fmse), 2) if val_fmse is not None else None,
        }
    elif arch == 'rainnet':
        model = RainNet(input_nc=3, output_nc=3, ngf=64, norm_layer=RAIN, use_dropout=True, use_attention=True)
        sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
        model.load_state_dict(strip_module_prefix(sd), strict=True)
        meta = {'epoch': 'Pre', 'val_fMSE': None}
    elif arch == 'ht_plus':
        model = HTGenerator(input_nc=3, output_nc=3, ksize=4, dim=256, nhead=2, num_layers=9, dim_forward=2, size=256)
        sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
        model.load_state_dict(strip_module_prefix(sd), strict=True)
        meta = {'epoch': 'Pre', 'val_fMSE': None}
    else:
        raise ValueError(f"Kiến trúc không hợp lệ: {arch}")

    model = model.to(device).eval()
    meta['total_params_M'] = round(sum(p.numel() for p in model.parameters()) / 1e6, 2)
    meta['trainable_params_M'] = round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 2)
    return model, meta

@torch.inference_mode()
def infer_raw_and_out(model, arch: str, c: Tensor, m: Tensor, use_amp: bool = True):
    """Chạy suy luận thống nhất trả về (raw, out) đều ở float32 trong [0, 1]."""
    if arch == 'identity':
        return c, c

    if arch == 'rainnet':
        # RainNet chuẩn hóa input ảnh về [-1, 1], mask ở [0, 1]; chạy float32 để tránh sai số RAIN
        c_m11 = c * 2.0 - 1.0
        raw_m11 = model(c_m11, m).float()
        raw = ((raw_m11 + 1.0) * 0.5).clamp(0.0, 1.0)
    elif arch == 'ht_plus':
        # HT_2H9L_allihd nhận [comp, mask] trong [0, 1] và xuất giá trị trong [0, 1] (clamp [0, 1])
        raw = model(torch.cat([c, m], dim=1)).float().clamp(0.0, 1.0)
    else:
        if use_amp and c.is_cuda:
            with torch.autocast('cuda', dtype=torch.float16):
                raw = model(torch.cat([c, m], dim=1))
        else:
            raw = model(torch.cat([c, m], dim=1))
        raw = raw.float().clamp(0.0, 1.0)

    out = (raw * m + c * (1.0 - m)).clamp(0.0, 1.0)
    return raw, out

print("Đã định nghĩa thành công 5 kiến trúc mạng (UNet, ImprovedUNet v2, ImprovedUNet v3, RainNet, HT+).")


Đã định nghĩa thành công 5 kiến trúc mạng (UNet, ImprovedUNet v2, ImprovedUNet v3, RainNet, HT+).


## Phần 3: Tái tạo Tập Test (7.404 ảnh), Xác minh Fingerprint, Nhóm Độ Khó & Bộ Chỉ Số
- Xác minh không rò rỉ dữ liệu Train/Val → Test và tính mã SHA-256 `fingerprint`.
- Tự động nạp bảng phân tầng 6 nhóm độ khó từ `iHarmony4_groups_train_test/test.csv` (`small/large` × `low/medium/high`).
- Định nghĩa 7 chỉ số đánh giá tính trực tiếp trên GPU: `MSE`, `fMSE`, `PSNR`, `fPSNR`, `SSIM`, `fSSIM`, và `fDeltaE76` (CIE Lab D65).

In [3]:
import csv
import hashlib
import json
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from pytorch_msssim import ssim as msssim_ssim

EXTS = {'.jpg', '.jpeg', '.png'}

def make_index(folder: Path):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS}

def build_records(seed: int = 42, val_fraction: float = 0.05):
    records = {'train': [], 'val': [], 'test': []}
    for subset in ARCHIVES:
        root = DATA / subset
        comp = make_index(root / 'composite_images')
        mask = make_index(root / 'masks')
        real = make_index(root / 'real_images')

        subsets = {}
        for split in ('train', 'test'):
            txt = root / f'{subset}_{split}.txt'
            rows = []
            for line in txt.read_text(encoding='utf-8-sig').splitlines():
                if not line.strip():
                    continue
                stem = Path(line.strip().replace('\\', '/').split()[0]).stem
                parts = stem.split('_')
                source_id = '_'.join(parts[:-2])
                mask_id = '_'.join(parts[:-1])
                rows.append(dict(
                    subset=subset, stem=stem, group=f'{subset}/{source_id}', full_id=f'{subset}/{stem}',
                    comp=str(comp[stem]), mask=str(mask[mask_id]), real=str(real[source_id]),
                ))
            subsets[split] = rows

        groups = sorted({r['group'] for r in subsets['train']})
        random.Random(seed).shuffle(groups)
        val_groups = set(groups[:max(1, round(len(groups) * val_fraction))])

        for r in subsets['train']:
            records['val' if r['group'] in val_groups else 'train'].append(r)
        records['test'].extend(subsets['test'])
    return records

records = build_records(seed=42, val_fraction=0.05)
split_ids = {s: [r['full_id'] for r in rows] for s, rows in records.items()}
recomputed_fp = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()

assert len(records['test']) == 7404, f"Sai số lượng test: {len(records['test'])}"
assert len(set(split_ids['train']) & set(split_ids['test'])) == 0, 'CẢNH BÁO: Rò rỉ Train -> Test!'
assert len(set(split_ids['val']) & set(split_ids['test'])) == 0, 'CẢNH BÁO: Rò rỉ Val -> Test!'
print(f"XÁC NHẬN SẠCH 100%: 0 ảnh Test lọt vào Train/Val ({len(records['test'])} ảnh test độc lập).")
print(f"Split SHA-256 fingerprint: {recomputed_fp}")

# Nạp thông tin 6 nhóm độ khó từ iHarmony4_groups_train_test/test.csv (nếu có)
GROUP_META = {}
GROUP_ORDER = ['small_low', 'small_medium', 'small_high', 'large_low', 'large_medium', 'large_high']
group_csv_path = GROUPS_DIR / 'test.csv'
if group_csv_path.is_file():
    with open(group_csv_path, 'r', encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            key = row.get('sample_key') or f"{row['dataset']}/{row['sample_id']}"
            GROUP_META[key] = {
                'mask_group': row['mask_group'],
                'color_group': row['color_group'],
                'joint_group': row['joint_group'],
                'mask_ratio': float(row['mask_ratio']),
                'input_deltaE76': float(row['input_deltaE76']),
            }
    matched = sum(1 for r in records['test'] if r['full_id'] in GROUP_META)
    print(f"Đã nạp phân tầng độ khó từ {group_csv_path.name}: khớp {matched}/{len(records['test'])} mẫu test.")
else:
    print(f"[THÔNG BÁO] Không tìm thấy {group_csv_path} — sẽ bỏ qua bảng phân tích 6 nhóm độ khó.")

class HarmonyTestDataset(Dataset):
    def __init__(self, rows, size: int = 256):
        self.rows, self.size = rows, size

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i: int):
        r = self.rows[i]
        target = (self.size, self.size)
        imgs = []
        for k, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[k]) as im:
                im = im.convert(mode)
                if im.size != target:
                    im = im.resize(target, Image.Resampling.NEAREST if k == 'mask' else Image.Resampling.BILINEAR)
                imgs.append(pil_to_tensor(im).float() / 255.0)
        c, m, y = imgs
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        return c, m, y, r['subset'], r['full_id']

# ==============================================================================
# CÁC HÀM TÍNH METRIC TRÊN GPU (fSSIM chuẩn HarmonyTransformer & CIE Lab DeltaE76)
# ==============================================================================
_SSIM_WINDOW_CACHE = {}

def _get_gaussian_window(window_size: int, channels: int, device: torch.device, dtype: torch.dtype) -> Tensor:
    key = (window_size, channels, device, dtype)
    if key not in _SSIM_WINDOW_CACHE:
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g1d = torch.exp(-(coords ** 2) / (2.0 * 1.5 ** 2))
        g1d = (g1d / g1d.sum()).unsqueeze(1)
        g2d = (g1d @ g1d.t()).unsqueeze(0).unsqueeze(0)
        win = g2d.expand(channels, 1, window_size, window_size).contiguous().to(device=device, dtype=dtype)
        _SSIM_WINDOW_CACHE[key] = win
    return _SSIM_WINDOW_CACHE[key]

def compute_fssim(img1: Tensor, img2: Tensor, mask: Tensor, window_size: int = 11) -> Tensor:
    """Foreground SSIM theo công thức chuẩn iHarmony4 / HarmonyTransformer (trả về vector shape (B,))."""
    C = img1.shape[1]
    win = _get_gaussian_window(window_size, C, img1.device, img1.dtype)
    pad = window_size // 2
    mu1 = F.conv2d(img1, win, padding=pad, groups=C)
    mu2 = F.conv2d(img2, win, padding=pad, groups=C)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(img1 * img1, win, padding=pad, groups=C) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, win, padding=pad, groups=C) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, win, padding=pad, groups=C) - mu1_mu2
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return (ssim_map * mask).sum(dim=(1, 2, 3)) / (C * mask.sum(dim=(1, 2, 3)).clamp_min(1.0))

def rgb_to_lab_d65(rgb: Tensor) -> Tensor:
    """Chuyển sRGB [0, 1] (B, 3, H, W) sang CIE Lab (D65)."""
    rgb = rgb.clamp(0.0, 1.0)
    lin = torch.where(rgb > 0.04045, ((rgb + 0.055) / 1.055).clamp_min(0.0).pow(2.4), rgb / 12.92)
    r, g, b = lin[:, 0], lin[:, 1], lin[:, 2]
    x = (0.4124564 * r + 0.3575761 * g + 0.1804375 * b) / 0.95047
    y = (0.2126729 * r + 0.7151522 * g + 0.0721750 * b) / 1.00000
    z = (0.0193339 * r + 0.1191920 * g + 0.9503041 * b) / 1.08883
    eps, kappa = 216.0 / 24389.0, 24389.0 / 27.0
    def _f(t):
        return torch.where(t > eps, t.clamp_min(1e-12).pow(1.0 / 3.0), (kappa * t + 16.0) / 116.0)
    fx, fy, fz = _f(x), _f(y), _f(z)
    L = 116.0 * fy - 16.0
    a = 500.0 * (fx - fy)
    b_ch = 200.0 * (fy - fz)
    return torch.stack([L, a, b_ch], dim=1)

def compute_batch_metrics(out: Tensor, y: Tensor, m: Tensor) -> dict[str, Tensor]:
    err = (out - y).square()
    mse = err.mean(dim=(1, 2, 3))
    fmse = (err * m).sum(dim=(1, 2, 3)) / (3.0 * m.sum(dim=(1, 2, 3)).clamp_min(1.0))

    lab_out = rgb_to_lab_d65(out)
    lab_y = rgb_to_lab_d65(y)
    de_map = torch.sqrt(((lab_out - lab_y) ** 2).sum(dim=1, keepdim=True).clamp_min(1e-12))
    f_delta_e = (de_map * m).sum(dim=(1, 2, 3)) / m.sum(dim=(1, 2, 3)).clamp_min(1.0)

    return {
        'MSE': mse * (255.0 ** 2),
        'fMSE': fmse * (255.0 ** 2),
        'PSNR': -10.0 * torch.log10(mse.clamp_min(1e-12)),
        'fPSNR': -10.0 * torch.log10(fmse.clamp_min(1e-12)),
        'SSIM': msssim_ssim(out, y, data_range=1.0, size_average=False),
        'fSSIM': compute_fssim(out, y, m, window_size=11),
        'fDeltaE76': f_delta_e,
    }


XÁC NHẬN SẠCH 100%: 0 ảnh Test lọt vào Train/Val (7404 ảnh test độc lập).
Split SHA-256 fingerprint: aad71bde27817a76a2ab959e795b0798dff7fe526c5fe16ec6663e43e79de6be
Đã nạp phân tầng độ khó từ test.csv: khớp 7404/7404 mẫu test.


## Phần 4: Chạy Đánh Giá Đầy Đủ Toàn Bộ 8 Mô Hình trong `runs/` (+ Input Composite)
- Tự động kiểm tra danh sách checkpoint trước khi chạy.
- Chấm lần lượt từng mô hình, tự giải phóng VRAM sau mỗi mô hình (không bị OOM trên T4).
- Xuất đầy đủ:
  - `evaluation_all_models/all_models_test_metrics.json`
  - `evaluation_all_models/summary_overall.csv`
  - `evaluation_all_models/summary_by_dataset.csv`
  - `evaluation_all_models/summary_by_difficulty_group.csv`
  - `evaluation_all_models/per_image_metrics/<slug>.csv` (7.404 dòng/mô hình)
  - `test_metrics_unified.json` & ảnh preview trong từng thư mục run.

In [4]:
import gc
import time
import pandas as pd
from torchvision.utils import save_image
from tqdm.auto import tqdm

# Bật True nếu muốn chấm thêm mốc epoch 30 của Masked L1 (nguồn rẽ nhánh của Branch B & C)
INCLUDE_MASKED_L1_30EP = False
# Lưu thêm test_metrics_unified.json vào từng thư mục run (không ghi đè test_metrics.json cũ)
SAVE_BACK_TO_RUN_DIRS = True

MODELS = {
    "0. Input Composite (No Harm)": {
        "slug": "00_input_composite",
        "short": "Input\nComposite",
        "arch": "identity",
        "path": None,
        "batch_size": 32,
        "amp": False,
    },
    "1. Baseline U-Net (Full L1)": {
        "slug": "01_unet_full_l1",
        "short": "U-Net\nFull L1",
        "arch": "unet",
        "path": resolve_run_file("unet_full_l1_all4_bs32_seed42/U-NET_Full_L1_checkpoint.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "2. Baseline U-Net (Masked L1)": {
        "slug": "02_unet_masked_l1",
        "short": "U-Net\nMasked L1",
        "arch": "unet",
        "path": resolve_run_file("unet_masked_l1_50ep_all4_bs32_seed42/best.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "3. U-Net + 1 PatchGAN (Branch B)": {
        "slug": "03_unet_branchB_1patchgan",
        "short": "U-Net +\n1-PatchGAN",
        "arch": "unet",
        "path": resolve_run_file("unet_branchB_1patchgan_50ep_all4_bs32_seed42/best.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "4. U-Net + 3 PatchGAN (Branch C)": {
        "slug": "04_unet_branchC_3patchgan",
        "short": "U-Net +\n3-PatchGAN",
        "arch": "unet",
        "path": resolve_run_file("unet_branchC_3patchgan_50ep_all4_bs32_seed42/best.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "5. Improved U-Net v2 (Hybrid)": {
        "slug": "05_improved_unet_v2_hybrid",
        "short": "Improved\nU-Net v2",
        "arch": "improved_unet_v2",
        "path": resolve_run_file("improved_unet_v2_hybrid_probe10ep_bs32_seed42/best.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "6. Improved U-Net v3 (DINOv2 + Lab)": {
        "slug": "06_improved_unet_v3_dinov2_lab",
        "short": "U-Net v3\nDINOv2+Lab",
        "arch": "improved_unet_v3",
        "path": resolve_run_file("improved_unet_v3_dinov2_lab_30ep_bs32_seed42/best.pt"),
        "batch_size": 32,
        "amp": True,
    },
    "7. RainNet Pretrained (CVPR 2021)": {
        "slug": "07_rainnet_pretrained",
        "short": "RainNet\nPretrained",
        "arch": "rainnet",
        "path": resolve_run_file("RainNet_Pretrained/RainNet_net_G_last.pth"),
        "batch_size": 32,
        "amp": False,
    },
    "8. HT+ Pretrained (HT_2H9L)": {
        "slug": "08_ht_plus_pretrained",
        "short": "HT+ 2H9L\nPretrained",
        "arch": "ht_plus",
        "path": resolve_run_file("ht_plus_pretrained/HT_2H9L_allihd_latest_net_G.pth"),
        "batch_size": 16,
        "amp": False,
    },
}

if INCLUDE_MASKED_L1_30EP:
    MODELS["2b. U-Net Masked L1 (30ep Milestone)"] = {
        "slug": "02b_unet_masked_l1_30ep",
        "short": "Masked L1\n(30ep)",
        "arch": "unet",
        "path": resolve_run_file("unet_masked_l1_50ep_all4_bs32_seed42/best_30ep.pt"),
        "batch_size": 32,
        "amp": True,
    }

print("Kiểm tra checkpoint trước khi đánh giá:")
for name, info in MODELS.items():
    p = info['path']
    if p is None:
        print(f"  [OK]   {name:<38} (Identity baseline)")
    elif p.is_file():
        mb = p.stat().st_size / (1024 ** 2)
        print(f"  [OK]   {name:<38} ({mb:6.1f} MB) -> {p}")
    else:
        print(f"  [MISS] {name:<38} -> Không thấy: {p}")

METRIC_KEYS = ['MSE', 'fMSE', 'PSNR', 'fPSNR', 'SSIM', 'fSSIM', 'fDeltaE76']
test_ds = HarmonyTestDataset(records['test'], size=256)
_loaders = {}
def get_test_loader(bs: int) -> DataLoader:
    if bs not in _loaders:
        _loaders[bs] = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    return _loaders[bs]

def aggregate_df(df: pd.DataFrame) -> dict:
    out = {'n': int(len(df))}
    for k in METRIC_KEYS:
        out[k] = round(float(df[k].mean()), 4)
    return out

all_results = {}
per_image_tables = {}

for name, info in MODELS.items():
    ck_path = info['path']
    if ck_path is not None and not ck_path.is_file():
        print(f"\n[BỎ QUA] Không tìm thấy checkpoint: {ck_path}")
        continue

    model, meta = load_harmonization_model(
        arch=info['arch'], ck_path=ck_path, device=DEVICE, expected_fp=recomputed_fp
    )
    loader = get_test_loader(info['batch_size'])

    # Warm-up 1 batch giả để đo thời gian suy luận chính xác
    if model is not None:
        dummy_c = torch.zeros(2, 3, 256, 256, device=DEVICE)
        dummy_m = torch.ones(2, 1, 256, 256, device=DEVICE)
        _ = infer_raw_and_out(model, info['arch'], dummy_c, dummy_m, use_amp=info['amp'])
        torch.cuda.synchronize()
        del dummy_c, dummy_m

    rows_out = []
    saved_preview = False
    total_infer_sec = 0.0
    total_imgs = 0

    for c, m, y, subsets, full_ids in tqdm(loader, desc=f"{name} (Ep {meta['epoch']})"):
        c = c.to(DEVICE, non_blocking=True)
        m = m.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        raw, out = infer_raw_and_out(model, info['arch'], c, m, use_amp=info['amp'])
        torch.cuda.synchronize()
        total_infer_sec += time.perf_counter() - t0
        total_imgs += c.shape[0]

        vals = compute_batch_metrics(out, y, m)
        vals_cpu = {k: v.detach().cpu().numpy() for k, v in vals.items()}

        for i in range(c.shape[0]):
            fid = full_ids[i]
            ginfo = GROUP_META.get(fid, {})
            row = {
                'full_id': fid,
                'subset': subsets[i],
                'mask_group': ginfo.get('mask_group', ''),
                'color_group': ginfo.get('color_group', ''),
                'joint_group': ginfo.get('joint_group', ''),
                'mask_ratio': ginfo.get('mask_ratio', float(m[i].mean().item())),
            }
            for k in METRIC_KEYS:
                row[k] = float(vals_cpu[k][i])
            rows_out.append(row)

        if not saved_preview and ck_path is not None:
            panels = torch.stack([c, m.expand_as(c), raw, out, y], dim=1).flatten(0, 1)
            preview_path = OUT_DIR / 'previews' / f"{info['slug']}_preview.png"
            save_image(panels[:30], str(preview_path), nrow=5)
            if SAVE_BACK_TO_RUN_DIRS:
                ep_tag = f"ep{meta['epoch']}" if isinstance(meta['epoch'], int) else 'pretrained'
                save_image(panels[:30], str(ck_path.parent / f"test_preview_{ep_tag}_unified.png"), nrow=5)
            saved_preview = True

    df = pd.DataFrame(rows_out)
    df.to_csv(OUT_DIR / 'per_image_metrics' / f"{info['slug']}.csv", index=False)
    per_image_tables[name] = df

    res = {'all': aggregate_df(df)}
    for sub in ARCHIVES:
        sub_df = df[df['subset'] == sub]
        if len(sub_df) > 0:
            res[sub] = aggregate_df(sub_df)

    if GROUP_META:
        res['by_mask_group'] = {
            g: aggregate_df(df[df['mask_group'] == g])
            for g in ('small', 'large') if (df['mask_group'] == g).any()
        }
        res['by_color_group'] = {
            g: aggregate_df(df[df['color_group'] == g])
            for g in ('low', 'medium', 'high') if (df['color_group'] == g).any()
        }
        res['by_joint_group'] = {
            g: aggregate_df(df[df['joint_group'] == g])
            for g in GROUP_ORDER if (df['joint_group'] == g).any()
        }

    ms_per_img = (total_infer_sec / max(total_imgs, 1)) * 1000.0
    res['_meta'] = {
        **meta,
        'slug': info['slug'],
        'arch': info['arch'],
        'checkpoint': str(ck_path) if ck_path is not None else None,
        'batch_size': info['batch_size'],
        'amp': info['amp'],
        'ms_per_img': round(ms_per_img, 3),
        'fps': round(1000.0 / max(ms_per_img, 1e-6), 1),
    }
    all_results[name] = res

    if SAVE_BACK_TO_RUN_DIRS and ck_path is not None:
        (ck_path.parent / 'test_metrics_unified.json').write_text(
            json.dumps(res, indent=2, ensure_ascii=False), encoding='utf-8'
        )

    del model
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# LƯU & IN CÁC BẢNG TỔNG HỢP
# ==============================================================================
(OUT_DIR / 'all_models_test_metrics.json').write_text(
    json.dumps(all_results, indent=2, ensure_ascii=False), encoding='utf-8'
)

base_fmse = all_results.get("0. Input Composite (No Harm)", {}).get('all', {}).get('fMSE')

# Bảng 1: Tổng hợp toàn tập Test (7.404 ảnh)
overall_rows = []
for name, res in all_results.items():
    m, a = res['_meta'], res['all']
    p_str = (
        f"{m['trainable_params_M']:.1f}/{m['total_params_M']:.1f}M"
        if m['trainable_params_M'] != m['total_params_M']
        else f"{m['total_params_M']:.2f}M"
    )
    gain_pct = ((base_fmse - a['fMSE']) / base_fmse * 100.0) if base_fmse else None
    overall_rows.append({
        'Model': name,
        'Ep': m['epoch'],
        'Params': p_str,
        'ms/img': m['ms_per_img'],
        'MSE': a['MSE'],
        'fMSE': a['fMSE'],
        'fMSE_Gain_%': round(gain_pct, 2) if gain_pct is not None else None,
        'PSNR': a['PSNR'],
        'fPSNR': a['fPSNR'],
        'SSIM': a['SSIM'],
        'fSSIM': a['fSSIM'],
        'fDeltaE76': a['fDeltaE76'],
    })

df_overall = pd.DataFrame(overall_rows)
df_overall.to_csv(OUT_DIR / 'summary_overall.csv', index=False)

print("\n" + "=" * 136)
print("BẢNG 1: TỔNG HỢP KẾT QUẢ TRÊN TOÀN BỘ 7.404 ẢNH TEST (iHarmony4 - 256x256, Ghép nền gốc)")
print("=" * 136)
print(
    f"{'MÔ HÌNH':<36} | {'Ep':>3} | {'Params':>11} | {'ms/im':>6} | {'MSE ↓':>7} | {'fMSE ↓':>8} | "
    f"{'ΔfMSE':>7} | {'PSNR ↑':>6} | {'fPSNR ↑':>7} | {'SSIM ↑':>6} | {'fSSIM ↑':>7} | {'fΔE76 ↓':>7}"
)
print("-" * 136)
for r in overall_rows:
    g_str = f"{r['fMSE_Gain_%']:+6.1f}%" if r['fMSE_Gain_%'] is not None else "    N/A"
    print(
        f"{r['Model']:<36} | {str(r['Ep']):>3} | {r['Params']:>11} | {r['ms/img']:>6.2f} | "
        f"{r['MSE']:>7.2f} | {r['fMSE']:>8.2f} | {g_str:>7} | {r['PSNR']:>6.2f} | "
        f"{r['fPSNR']:>7.2f} | {r['SSIM']:>6.4f} | {r['fSSIM']:>7.4f} | {r['fDeltaE76']:>7.2f}"
    )
print("=" * 136)

# Bảng 2: Chi tiết theo 4 Sub-dataset
sub_rows = []
for name, res in all_results.items():
    for sub in ARCHIVES + ['all']:
        if sub in res:
            sub_rows.append({'Model': name, 'Dataset': sub.upper() if sub == 'all' else sub, **res[sub]})
df_by_dataset = pd.DataFrame(sub_rows)
df_by_dataset.to_csv(OUT_DIR / 'summary_by_dataset.csv', index=False)

print("\nBẢNG 2: fMSE ↓ / PSNR ↑ THEO TỪNG SUB-DATASET")
print("-" * 116)
print(f"{'MÔ HÌNH':<36} | {'HCOCO (4283)':>14} | {'HAdobe5k (2160)':>15} | {'HFlickr (828)':>14} | {'Hday2night (133)':>16} | {'ALL (7404)':>14}")
print("-" * 116)
for name, res in all_results.items():
    cells_str = [f"{res[s]['fMSE']:6.1f} / {res[s]['PSNR']:5.2f}" for s in ARCHIVES + ['all']]
    print(f"{name:<36} | {cells_str[0]:>14} | {cells_str[1]:>15} | {cells_str[2]:>14} | {cells_str[3]:>16} | {cells_str[4]:>14}")
print("-" * 116)

# Bảng 3: Chi tiết theo 6 nhóm độ khó
if GROUP_META:
    grp_rows = []
    for name, res in all_results.items():
        for g in GROUP_ORDER:
            if g in res.get('by_joint_group', {}):
                grp_rows.append({'Model': name, 'Joint_Group': g, **res['by_joint_group'][g]})
    df_by_group = pd.DataFrame(grp_rows)
    df_by_group.to_csv(OUT_DIR / 'summary_by_difficulty_group.csv', index=False)

    print("\nBẢNG 3: fMSE ↓ THEO 6 NHÓM ĐỘ KHÓ (mask_size × color_shift_DeltaE76)")
    print("-" * 116)
    print(f"{'MÔ HÌNH':<36} | " + " | ".join(f"{g:>11}" for g in GROUP_ORDER))
    print("-" * 116)
    for name, res in all_results.items():
        jg = res.get('by_joint_group', {})
        vals_str = " | ".join(f"{jg[g]['fMSE']:>11.2f}" if g in jg else f"{'N/A':>11}" for g in GROUP_ORDER)
        print(f"{name:<36} | {vals_str}")
    print("-" * 116)

print(f"\nĐã xuất toàn bộ kết quả JSON & CSV vào: {OUT_DIR}")


Kiểm tra checkpoint trước khi đánh giá:
  [OK]   0. Input Composite (No Harm)           (Identity baseline)
  [OK]   1. Baseline U-Net (Full L1)            (  29.6 MB) -> /content/drive/MyDrive/ImageHarmonization/runs/unet_full_l1_all4_bs32_seed42/U-NET_Full_L1_checkpoint.pt
  [OK]   2. Baseline U-Net (Masked L1)          (  29.6 MB) -> /content/drive/MyDrive/ImageHarmonization/runs/unet_masked_l1_50ep_all4_bs32_seed42/best.pt
  [OK]   3. U-Net + 1 PatchGAN (Branch B)       (  40.3 MB) -> /content/drive/MyDrive/ImageHarmonization/runs/unet_branchB_1patchgan_50ep_all4_bs32_seed42/best.pt
  [OK]   4. U-Net + 3 PatchGAN (Branch C)       (  61.5 MB) -> /content/drive/MyDrive/ImageHarmonization/runs/unet_branchC_3patchgan_50ep_all4_bs32_seed42/best.pt
  [OK]   5. Improved U-Net v2 (Hybrid)          ( 177.9 MB) -> /content/drive/MyDrive/ImageHarmonization/runs/improved_unet_v2_hybrid_probe10ep_bs32_seed42/best.pt
  [OK]   6. Improved U-Net v3 (DINOv2 + Lab)    ( 127.1 MB) -> /content/drive/M

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 309MB/s]



BẢNG 1: TỔNG HỢP KẾT QUẢ TRÊN TOÀN BỘ 7.404 ẢNH TEST (iHarmony4 - 256x256, Ghép nền gốc)
MÔ HÌNH                              |  Ep |      Params |  ms/im |   MSE ↓ |   fMSE ↓ |   ΔfMSE | PSNR ↑ | fPSNR ↑ | SSIM ↑ | fSSIM ↑ | fΔE76 ↓
----------------------------------------------------------------------------------------------------------------------------------------
0. Input Composite (No Harm)         |   - |       0.00M |   0.01 |  172.90 |  1359.67 |   +0.0% |  31.43 |   19.05 | 0.9731 |  0.8244 |   16.13
1. Baseline U-Net (Full L1)          |  39 |       7.76M |   4.68 |   60.59 |   586.10 |  +56.9% |  34.61 |   22.55 | 0.9815 |  0.8786 |   11.69
2. Baseline U-Net (Masked L1)        |  39 |       7.76M |   4.79 |   41.67 |   401.16 |  +70.5% |  35.95 |   24.16 | 0.9845 |  0.9055 |   10.05
3. U-Net + 1 PatchGAN (Branch B)     |  38 |       7.76M |   4.81 |   41.98 |   401.50 |  +70.5% |  35.92 |   24.12 | 0.9843 |  0.9041 |   10.20
4. U-Net + 3 PatchGAN (Branch C)     |  38 |    

## Phần 5: Trực quan hóa Lưới Ảnh Đối Chiếu Tất Cả Các Mô Hình
Tự động ghép lưới ảnh gồm `Input Composite | Mask | Tất cả các mô hình đã đánh giá | Ground Truth`:
1. **Lưới 1 (`comparison_all_models_4datasets.png`):** 2 ảnh ngẫu nhiên từ mỗi sub-dataset (8 hàng).
2. **Lưới 2 (`comparison_all_models_hardest_large_high.png`):** Các mẫu khó nhất thuộc nhóm `large_high` (vùng ghép lớn + độ lệch màu cao) từ 4 sub-dataset để dễ nhìn rõ sự khác biệt giữa các mô hình.

In [5]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

def render_comparison_grid(chosen_rows, save_path: Path, title_prefix: str = ""):
    active_keys = [
        k for k, info in MODELS.items()
        if info['arch'] != 'identity' and info['path'] is not None and info['path'].is_file()
    ]
    assert len(active_keys) > 0, "Không có checkpoint mô hình nào tồn tại để vẽ lưới so sánh!"

    IMG_SIZE = 256
    ds_vis = HarmonyTestDataset(chosen_rows, IMG_SIZE)
    cs, ms, ys, subs, fids = zip(*[ds_vis[i] for i in range(len(ds_vis))])
    c_b = torch.stack(cs).to(DEVICE)
    m_b = torch.stack(ms).to(DEVICE)
    y_b = torch.stack(ys).to(DEVICE)

    outs = {}
    ep_labels = {}
    for key in active_keys:
        info = MODELS[key]
        net, meta = load_harmonization_model(info['arch'], info['path'], DEVICE)
        _, out = infer_raw_and_out(net, info['arch'], c_b, m_b, use_amp=info['amp'])
        outs[key] = out.cpu()
        ep_labels[key] = f"Ep {meta['epoch']}" if isinstance(meta['epoch'], int) else str(meta['epoch'])
        del net
        torch.cuda.empty_cache()

    cols_tensors = [c_b.cpu(), m_b.expand_as(c_b).cpu()] + [outs[k] for k in active_keys] + [y_b.cpu()]
    col_titles = (
        ['Input\nComposite', 'Mask']
        + [f"{MODELS[k]['short']}\n({ep_labels[k]})" for k in active_keys]
        + ['Ground\nTruth']
    )

    ncols = len(col_titles)
    nrows = len(chosen_rows)
    panels = torch.stack(cols_tensors, dim=1).flatten(0, 1)

    PADDING = 4
    grid = make_grid(panels, nrow=ncols, padding=PADDING, pad_value=1.0)

    fig, ax = plt.subplots(figsize=(ncols * 2.7, nrows * 2.7 + 1.4))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis('off')

    step_x = IMG_SIZE + PADDING
    for idx, title in enumerate(col_titles):
        ax.text(
            PADDING + idx * step_x + IMG_SIZE / 2, -12, title,
            ha='center', va='bottom', fontsize=10.5, fontweight='bold', color='#111111'
        )

    step_y = IMG_SIZE + PADDING
    for row_idx, r in enumerate(chosen_rows):
        g_tag = GROUP_META.get(r['full_id'], {}).get('joint_group', '')
        lbl = f"{r['subset']}\n{g_tag}" if g_tag else f"{r['subset']}\n#{row_idx + 1}"
        ax.text(
            -12, PADDING + row_idx * step_y + IMG_SIZE / 2, lbl,
            ha='right', va='center', fontsize=10, fontweight='bold', color='#222222'
        )

    if title_prefix:
        fig.suptitle(title_prefix, fontsize=14, fontweight='bold', y=1.01)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Đã lưu lưới so sánh ({nrows} hàng × {ncols} cột) tại: {save_path}")

# 1) Lưới ngẫu nhiên 2 ảnh / sub-dataset (8 hàng)
rng = random.Random(123)
chosen_random = []
for subset in ARCHIVES:
    sub_rows = [r for r in records['test'] if r['subset'] == subset]
    chosen_random.extend(rng.sample(sub_rows, 2))

render_comparison_grid(
    chosen_random,
    OUT_DIR / "comparison_all_models_4datasets.png",
    title_prefix="So sánh trực quan tất cả mô hình (2 mẫu ngẫu nhiên / dataset)",
)

# 2) Lưới mẫu khó nhất (nhóm large_high, sắp theo độ lệch màu input_deltaE76 cao)
if GROUP_META:
    chosen_hard = []
    for subset in ARCHIVES:
        hard_rows = [
            r for r in records['test']
            if r['subset'] == subset and GROUP_META.get(r['full_id'], {}).get('joint_group') == 'large_high'
        ]
        hard_rows.sort(key=lambda r: GROUP_META[r['full_id']]['input_deltaE76'], reverse=True)
        if len(hard_rows) >= 2:
            # Lấy mẫu ở vị trí phân vị 10% và 30% để tránh outlier cực đoan nhưng vẫn rất khó
            chosen_hard.extend([hard_rows[len(hard_rows) // 10], hard_rows[len(hard_rows) // 3]])
        else:
            chosen_hard.extend(hard_rows)
    if chosen_hard:
        render_comparison_grid(
            chosen_hard,
            OUT_DIR / "comparison_all_models_hardest_large_high.png",
            title_prefix="So sánh trực quan trên nhóm khó nhất (large_high: vùng ghép lớn & lệch màu cao)",
        )


Đã lưu lưới so sánh (8 hàng × 11 cột) tại: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/comparison_all_models_4datasets.png


Đã lưu lưới so sánh (8 hàng × 11 cột) tại: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/comparison_all_models_hardest_large_high.png


## Phần 6: Biểu đồ Phân tích Định lượng (Dùng trực tiếp cho Báo cáo)
1. So sánh **fMSE ↓** và **PSNR ↑** theo từng sub-dataset (`HCOCO`, `HAdobe5k`, `HFlickr`, `Hday2night`, `ALL`).
2. Heatmap **fMSE ↓** theo **6 nhóm độ khó** (`small_low` → `large_high`).
3. Biểu đồ đánh đổi **Chất lượng (fMSE ↓) vs Số tham số & Tốc độ suy luận (ms/ảnh)**.

In [6]:
import numpy as np
import matplotlib.pyplot as plt

# Chỉ vẽ các mô hình thực tế (bỏ Input Composite khỏi biểu đồ scale chi tiết để dễ phân biệt độ chênh lệch)
model_names = [k for k in all_results.keys() if all_results[k]['_meta']['arch'] != 'identity']
short_labels = [MODELS[k]['short'].replace('\n', ' ') for k in model_names]

# 1. Grouped Bar Chart: fMSE & PSNR theo từng sub-dataset
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
datasets_plot = ARCHIVES + ['all']
ds_labels = ARCHIVES + ['ALL']
x = np.arange(len(datasets_plot))
width = 0.8 / max(len(model_names), 1)
cmap = plt.get_cmap('tab10')

for idx, (mname, slbl) in enumerate(zip(model_names, short_labels)):
    fmse_vals = [all_results[mname][d]['fMSE'] for d in datasets_plot]
    psnr_vals = [all_results[mname][d]['PSNR'] for d in datasets_plot]
    offset = (idx - (len(model_names) - 1) / 2.0) * width
    axes[0].bar(x + offset, fmse_vals, width, label=slbl, color=cmap(idx % 10))
    axes[1].bar(x + offset, psnr_vals, width, label=slbl, color=cmap(idx % 10))

axes[0].set_title('fMSE theo từng Sub-dataset (Thấp hơn = Tốt hơn ↓)', fontsize=12, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(ds_labels, fontweight='bold')
axes[0].set_ylabel('fMSE (thang 0..255)'); axes[0].grid(axis='y', alpha=0.3)

axes[1].set_title('PSNR theo từng Sub-dataset (Cao hơn = Tốt hơn ↑)', fontsize=12, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(ds_labels, fontweight='bold')
axes[1].set_ylabel('PSNR (dB)'); axes[1].set_ylim(bottom=28); axes[1].grid(axis='y', alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=min(4, len(model_names)), bbox_to_anchor=(0.5, -0.06), fontsize=9.5)
plt.tight_layout()
chart1_path = OUT_DIR / 'charts' / 'metrics_by_dataset.png'
plt.savefig(chart1_path, bbox_inches='tight', dpi=200)
plt.show()
print('Đã lưu:', chart1_path)

# 2. Heatmap fMSE theo 6 nhóm độ khó
if GROUP_META:
    mat = np.array([
        [all_results[m]['by_joint_group'][g]['fMSE'] for g in GROUP_ORDER]
        for m in model_names
    ])
    fig, ax = plt.subplots(figsize=(11, max(4.5, len(model_names) * 0.65)))
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(np.arange(len(GROUP_ORDER)))
    ax.set_xticklabels(GROUP_ORDER, fontsize=10, fontweight='bold')
    ax.set_yticks(np.arange(len(model_names)))
    ax.set_yticklabels(short_labels, fontsize=10, fontweight='bold')
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            color = 'white' if val > (mat.min() + 0.6 * (mat.max() - mat.min())) else 'black'
            ax.text(j, i, f"{val:.1f}", ha='center', va='center', color=color, fontsize=9.5, fontweight='bold')
    ax.set_title('Heatmap fMSE ↓ theo 6 nhóm độ khó (mask_size × color_shift)', fontsize=12, fontweight='bold', pad=12)
    plt.colorbar(im, ax=ax, label='fMSE ↓')
    plt.tight_layout()
    chart2_path = OUT_DIR / 'charts' / 'fmse_heatmap_difficulty_groups.png'
    plt.savefig(chart2_path, bbox_inches='tight', dpi=200)
    plt.show()
    print('Đã lưu:', chart2_path)

# 3. Trade-off Scatter: fMSE vs Latency (ms/img), kích thước điểm tỉ lệ với Total Params
fig, ax = plt.subplots(figsize=(10, 6))
for idx, (mname, slbl) in enumerate(zip(model_names, short_labels)):
    m = all_results[mname]['_meta']
    fmse = all_results[mname]['all']['fMSE']
    lat = m['ms_per_img']
    params = m['total_params_M']
    ax.scatter(lat, fmse, s=max(params * 18, 80), alpha=0.82, color=cmap(idx % 10), edgecolors='black', linewidth=1.0)
    ax.annotate(
        f"{slbl} ({params:.1f}M)",
        (lat, fmse),
        textcoords='offset points', xytext=(8, 4), fontsize=9, fontweight='bold',
    )
ax.set_xlabel('Thời gian suy luận (ms / ảnh trên GPU) ↓', fontsize=11, fontweight='bold')
ax.set_ylabel('Test fMSE trên 7.404 ảnh ↓', fontsize=11, fontweight='bold')
ax.set_title('Đánh đổi Hiệu quả: Test fMSE ↓ vs Tốc độ suy luận (cỡ hình tròn = Tổng số tham số)', fontsize=11.5, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
chart3_path = OUT_DIR / 'charts' / 'tradeoff_fmse_speed_params.png'
plt.savefig(chart3_path, bbox_inches='tight', dpi=200)
plt.show()
print('Đã lưu:', chart3_path)


Đã lưu: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/charts/metrics_by_dataset.png


Đã lưu: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/charts/fmse_heatmap_difficulty_groups.png


Đã lưu: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/charts/tradeoff_fmse_speed_params.png


## Phần 7: Đánh Giá Riêng Cho RainNet (Không Gian Lận vs Đúng Cách Đo Repo Gốc `junleen/RainNet`)

Phần này chạy **độc lập** (chỉ cần chạy xong **Phần 1, 2, 3** — không cần chạy lại toàn bộ Phần 4 nếu đã có sẵn `evaluation_all_models/all_models_test_metrics.json` trên Drive), dùng **duy nhất checkpoint `RainNet_Pretrained/RainNet_net_G_last.pth` (256×256)** hiện có trong `runs/` (không tải thêm file ngoài) và chạy mất **~1 phút trên GPU T4**.

### Vì sao cần bảng đối chiếu riêng này cho RainNet?
Trong repo gốc `junleen/RainNet` (`data/iharmony4_dataset.py`, dòng 72), trước khi đưa ảnh vào mạng, dataloader luôn chạy:
```python
comp = self._compose(comp, mask, real)  # comp = comp * mask + real * (1 - mask)
```
Tức là cách đo gốc của RainNet **dùng ảnh Ground-Truth `real` để thay toàn bộ vùng nền `1 - mask`** vào input `comp`. Khi tắt bước này (**Blind Test — không gian lận**), điểm RainNet giảm rõ rệt do hai yếu tố tách biệt: (1) mất lợi thế `MSE_bg = 0` và (2) mạng chỉ nhận 3 kênh RGB nên rất nhạy với vòng viền 1–2 pixel bị loang màu sau resize bilinear.

### 4 chế độ đo trên cùng checkpoint `RainNet_net_G_last.pth` (7.404 ảnh Test):
| Mã | Tên chế độ | Input `comp_in` | Ghép nền output | Kiểu chấm | Ý nghĩa |
| :--- | :--- | :--- | :--- | :---: | :--- |
| **R1** | **Blind (KHÔNG gian lận)** | `comp` gốc | `comp` gốc | Float32 | **Công bằng 100%** (cùng chuẩn với U-Net, PatchGAN, v2, v3, HT+) |
| **R2** | **GT-BG Input + Fair Comp BG** | `comp*m + real*(1-m)` | `comp` gốc | Float32 | Tách riêng tác động viền 2px lên `fMSE` (vẫn giữ sai số nền `comp` thật) |
| **R3** | **Official `junleen/RainNet` (Float32)** | `comp*m + real*(1-m)` | `real` (`MSE_bg=0`) | Float32 | Đúng cách đo `Inference_RainNet.ipynb` Cell 5 (`fMSE 531.44 / PSNR 35.61`) |
| **R4** | **Official `junleen/RainNet` (`evaluate.py`)** | `comp*m + real*(1-m)` | `real` (`MSE_bg=0`) | `tensor2im` uint8 | Đúng 100% `evaluate.py` gốc (`RainNet_test_results_7404.csv`: `536.63 / 35.57`) |


In [8]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from tqdm.auto import tqdm

# ==============================================================================
# ĐÁNH GIÁ RIÊNG CHO RAINNET (CHỈ DÙNG CHECKPOINT 256x256 CÓ SẴN TRONG runs/)
# ==============================================================================
RAINNET_AUDIT_DIR = OUT_DIR / "rainnet_audit"
RAINNET_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(RAINNET_AUDIT_DIR / "per_image_metrics").mkdir(parents=True, exist_ok=True)

ckpt_rn256 = resolve_run_file("RainNet_Pretrained/RainNet_net_G_last.pth")
assert ckpt_rn256.is_file(), f"Không tìm thấy checkpoint RainNet 256x256 tại: {ckpt_rn256}"
print("Sử dụng checkpoint RainNet gốc:", ckpt_rn256)

RAINNET_PROTOCOLS = [
    {
        "name": "R1. RainNet — Blind (KHÔNG gian lận)",
        "slug": "R1_rainnet_blind_fair",
        "short": "R1: Blind\n(KHÔNG gian lận)",
        "gt_bg_input": False,
        "gt_bg_output": False,
        "tensor2im_uint8": False,
        "fair": True,
    },
    {
        "name": "R2. RainNet — GT-BG In + Fair Comp BG",
        "slug": "R2_rainnet_gtbg_in_fair_bg",
        "short": "R2: GT-BG In\n+ Fair Comp BG",
        "gt_bg_input": True,
        "gt_bg_output": False,
        "tensor2im_uint8": False,
        "fair": False,
    },
    {
        "name": "R3. RainNet — Official Repo (Float32, GT-BG*)",
        "slug": "R3_rainnet_official_float32",
        "short": "R3: Official\nFloat32 (GT-BG*)",
        "gt_bg_input": True,
        "gt_bg_output": True,
        "tensor2im_uint8": False,
        "fair": False,
    },
    {
        "name": "R4. RainNet — Official evaluate.py (uint8, GT-BG*)",
        "slug": "R4_rainnet_official_evaluate_uint8",
        "short": "R4: Official\nevaluate.py (uint8*)",
        "gt_bg_input": True,
        "gt_bg_output": True,
        "tensor2im_uint8": True,
        "fair": False,
    },
]

@torch.inference_mode()
def infer_rainnet_mode(model, c: Tensor, m: Tensor, y: Tensor, gt_bg_input: bool, gt_bg_output: bool, tensor2im_uint8: bool):
    """
    Chạy suy luận RainNet đúng công thức junleen/RainNet:
      - Normalize input sang [-1, 1] giống transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
      - Nếu gt_bg_input=True: thực hiện đúng dòng 72 Iharmony4Dataset._compose(comp, mask, real)
      - Nếu tensor2im_uint8=True: tái hiện đúng util.tensor2im (cắt cụt .astype(uint8) không làm tròn)
    Trả về (out_eval, y_eval) đều ở float32 trong [0, 1].
    """
    c_m11 = (c - 0.5) / 0.5
    y_m11 = (y - 0.5) / 0.5
    c_in_m11 = (c_m11 * m + y_m11 * (1.0 - m)) if gt_bg_input else c_m11

    raw_m11 = model(c_in_m11, m).float()
    bg_m11 = y_m11 if gt_bg_output else c_m11
    out_m11 = raw_m11 * m + bg_m11 * (1.0 - m)

    if tensor2im_uint8:
        # Đúng từng dòng RainNet/util/util.py: ((x + 1) / 2.0 * 255.0).astype(np.uint8)
        out_u8 = ((out_m11 + 1.0) * 0.5 * 255.0).clamp(0.0, 255.0).to(torch.uint8)
        y_u8 = ((y_m11 + 1.0) * 0.5 * 255.0).clamp(0.0, 255.0).to(torch.uint8)
        return out_u8.float() / 255.0, y_u8.float() / 255.0

    out_01 = ((out_m11 + 1.0) * 0.5).clamp(0.0, 1.0)
    return out_01, y

RN_METRIC_KEYS = ["MSE", "fMSE", "PSNR", "fPSNR", "SSIM", "fSSIM", "fDeltaE76"]
rn_test_ds = HarmonyTestDataset(records["test"], size=256)
rn_loader = DataLoader(rn_test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

def _agg_rn_df(df: pd.DataFrame) -> dict:
    return {"n": int(len(df)), **{k: round(float(df[k].mean()), 4) for k in RN_METRIC_KEYS}}

# Nạp kết quả tham chiếu từ Phần 4 (trong RAM hoặc đọc từ JSON trên Drive nếu chạy Phần 7 độc lập)
ref_all_results = globals().get("all_results")
ref_json_path = OUT_DIR / "all_models_test_metrics.json"
if (not ref_all_results) and ref_json_path.is_file():
    ref_all_results = json.loads(ref_json_path.read_text(encoding="utf-8"))
    print(f"Đã nạp bảng tham chiếu các mô hình khác từ: {ref_json_path}")

# Nạp RainNet 1 lần duy nhất cho cả 4 chế độ (tổng thời gian chạy ~45-60 giây trên T4)
rn_model, rn_meta = load_harmonization_model("rainnet", ckpt_rn256, DEVICE)

# Warm-up
dummy_c = torch.zeros(2, 3, 256, 256, device=DEVICE)
dummy_m = torch.ones(2, 1, 256, 256, device=DEVICE)
_ = infer_rainnet_mode(rn_model, dummy_c, dummy_m, dummy_c, False, False, False)
torch.cuda.synchronize()
del dummy_c, dummy_m

rainnet_audit_results = {}
rainnet_audit_dfs = {}

for cfg_p in RAINNET_PROTOCOLS:
    rows_out = []
    total_sec, total_imgs = 0.0, 0
    for c, m, y, subsets, full_ids in tqdm(rn_loader, desc=cfg_p["name"]):
        c = c.to(DEVICE, non_blocking=True)
        m = m.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out_eval, y_eval = infer_rainnet_mode(
            rn_model, c, m, y,
            gt_bg_input=cfg_p["gt_bg_input"],
            gt_bg_output=cfg_p["gt_bg_output"],
            tensor2im_uint8=cfg_p["tensor2im_uint8"],
        )
        torch.cuda.synchronize()
        total_sec += time.perf_counter() - t0
        total_imgs += c.shape[0]

        vals = compute_batch_metrics(out_eval, y_eval, m)
        vals_cpu = {k: v.detach().cpu().numpy() for k, v in vals.items()}
        for i in range(c.shape[0]):
            fid = full_ids[i]
            ginfo = GROUP_META.get(fid, {})
            row = {
                "full_id": fid,
                "subset": subsets[i],
                "mask_group": ginfo.get("mask_group", ""),
                "color_group": ginfo.get("color_group", ""),
                "joint_group": ginfo.get("joint_group", ""),
                "mask_ratio": ginfo.get("mask_ratio", float(m[i].mean().item())),
            }
            for k in RN_METRIC_KEYS:
                row[k] = float(vals_cpu[k][i])
            rows_out.append(row)

    df_p = pd.DataFrame(rows_out)
    df_p.to_csv(RAINNET_AUDIT_DIR / "per_image_metrics" / f"{cfg_p['slug']}.csv", index=False)
    rainnet_audit_dfs[cfg_p["name"]] = df_p

    res_p = {"all": _agg_rn_df(df_p)}
    for sub in ARCHIVES:
        sub_df = df_p[df_p["subset"] == sub]
        if len(sub_df) > 0:
            res_p[sub] = _agg_rn_df(sub_df)
    if GROUP_META:
        res_p["by_mask_group"] = {
            g: _agg_rn_df(df_p[df_p["mask_group"] == g])
            for g in ("small", "large") if (df_p["mask_group"] == g).any()
        }
        res_p["by_color_group"] = {
            g: _agg_rn_df(df_p[df_p["color_group"] == g])
            for g in ("low", "medium", "high") if (df_p["color_group"] == g).any()
        }
        res_p["by_joint_group"] = {
            g: _agg_rn_df(df_p[df_p["joint_group"] == g])
            for g in GROUP_ORDER if (df_p["joint_group"] == g).any()
        }
    ms_img = (total_sec / max(total_imgs, 1)) * 1000.0
    res_p["_meta"] = {
        "slug": cfg_p["slug"],
        "checkpoint": str(ckpt_rn256),
        "gt_bg_input": cfg_p["gt_bg_input"],
        "gt_bg_output": cfg_p["gt_bg_output"],
        "tensor2im_uint8": cfg_p["tensor2im_uint8"],
        "fair_blind": cfg_p["fair"],
        "total_params_M": rn_meta["total_params_M"],
        "ms_per_img": round(ms_img, 3),
    }
    rainnet_audit_results[cfg_p["name"]] = res_p

# ==============================================================================
# IN & XUẤT BẢNG ĐỐI CHIẾU CHUYÊN SÂU RAINNET (BLIND vs CÁCH ĐO GỐC)
# ==============================================================================
(RAINNET_AUDIT_DIR / "rainnet_audit_metrics.json").write_text(
    json.dumps(rainnet_audit_results, indent=2, ensure_ascii=False), encoding="utf-8"
)

base_fmse_rn = 1359.67
ref_entries = []
if ref_all_results:
    if "0. Input Composite (No Harm)" in ref_all_results:
        base_fmse_rn = ref_all_results["0. Input Composite (No Harm)"]["all"]["fMSE"]
        ref_entries.append(("Ref0. Input Composite (No Harm)", ref_all_results["0. Input Composite (No Harm)"], "Blind"))
    for k_ref in [
        "1. Baseline U-Net (Full L1)",
        "2. Baseline U-Net (Masked L1)",
        "6. Improved U-Net v3 (DINOv2 + Lab)",
        "8. HT+ Pretrained (HT_2H9L)",
    ]:
        if k_ref in ref_all_results:
            ref_entries.append((f"Ref. {k_ref}", ref_all_results[k_ref], "Blind"))

combined_rows = []
for name, res in rainnet_audit_results.items():
    m_info, a = res["_meta"], res["all"]
    tag = "Blind" if m_info["fair_blind"] else "GT-BG*"
    gain = (base_fmse_rn - a["fMSE"]) / base_fmse_rn * 100.0
    combined_rows.append({
        "Protocol": name, "Mode": tag, "ms/img": m_info["ms_per_img"],
        "MSE": a["MSE"], "fMSE": a["fMSE"], "fMSE_Gain_%": round(gain, 2),
        "PSNR": a["PSNR"], "fPSNR": a["fPSNR"], "SSIM": a["SSIM"],
        "fSSIM": a["fSSIM"], "fDeltaE76": a["fDeltaE76"],
    })
for rname, rres, rtag in ref_entries:
    a = rres["all"]
    gain = (base_fmse_rn - a["fMSE"]) / base_fmse_rn * 100.0
    combined_rows.append({
        "Protocol": rname, "Mode": rtag, "ms/img": rres.get("_meta", {}).get("ms_per_img", 0.0),
        "MSE": a["MSE"], "fMSE": a["fMSE"], "fMSE_Gain_%": round(gain, 2),
        "PSNR": a["PSNR"], "fPSNR": a["fPSNR"], "SSIM": a["SSIM"],
        "fSSIM": a["fSSIM"], "fDeltaE76": a["fDeltaE76"],
    })

pd.DataFrame(combined_rows).to_csv(RAINNET_AUDIT_DIR / "rainnet_audit_summary.csv", index=False)

print("\n" + "=" * 144)
print("BẢNG R1: ĐÁNH GIÁ RIÊNG RAINNET (256x256) — KHÔNG GIAN LẬN (Blind) vs ĐÚNG CÁCH ĐO REPO GỐC (GT-BG*)")
print("=" * 144)
print(
    f"{'CẤU HÌNH ĐÁNH GIÁ':<50} | {'Chế độ':>6} | {'ms/im':>6} | {'MSE ↓':>7} | {'fMSE ↓':>8} | "
    f"{'ΔfMSE':>7} | {'PSNR ↑':>6} | {'fPSNR ↑':>7} | {'SSIM ↑':>6} | {'fSSIM ↑':>7} | {'fΔE76 ↓':>7}"
)
print("-" * 144)
for idx_r, r in enumerate(combined_rows):
    if idx_r == len(rainnet_audit_results) and ref_entries:
        print("-" * 144)
    print(
        f"{r['Protocol']:<50} | {r['Mode']:>6} | {r['ms/img']:>6.2f} | "
        f"{r['MSE']:>7.2f} | {r['fMSE']:>8.2f} | {r['fMSE_Gain_%']:+6.1f}% | {r['PSNR']:>6.2f} | "
        f"{r['fPSNR']:>7.2f} | {r['SSIM']:>6.4f} | {r['fSSIM']:>7.4f} | {r['fDeltaE76']:>7.2f}"
    )
print("=" * 144)

# Bảng R2: Theo 4 sub-dataset
sub_rows_rn = []
print("\nBẢNG R2: fMSE ↓ / PSNR ↑ CỦA RAINNET THEO TỪNG SUB-DATASET")
print("-" * 128)
print(f"{'CẤU HÌNH ĐÁNH GIÁ':<50} | {'HCOCO (4283)':>14} | {'HAdobe5k (2160)':>15} | {'HFlickr (828)':>14} | {'Hday2night (133)':>16} | {'ALL (7404)':>14}")
print("-" * 128)
for name, res in list(rainnet_audit_results.items()) + [(rn, rr) for rn, rr, _ in ref_entries]:
    for sub in ARCHIVES + ["all"]:
        if sub in res:
            sub_rows_rn.append({"Protocol": name, "Dataset": sub.upper() if sub == "all" else sub, **res[sub]})
    cells = [f"{res[s]['fMSE']:6.1f} / {res[s]['PSNR']:5.2f}" for s in ARCHIVES + ["all"]]
    print(f"{name:<50} | {cells[0]:>14} | {cells[1]:>15} | {cells[2]:>14} | {cells[3]:>16} | {cells[4]:>14}")
print("-" * 128)
pd.DataFrame(sub_rows_rn).to_csv(RAINNET_AUDIT_DIR / "rainnet_audit_by_dataset.csv", index=False)

# Bảng R3: Theo 6 nhóm độ khó
if GROUP_META:
    grp_rows_rn = []
    print("\nBẢNG R3: fMSE ↓ CỦA RAINNET THEO 6 NHÓM ĐỘ KHÓ (mask_size × color_shift)")
    print("-" * 130)
    print(f"{'CẤU HÌNH ĐÁNH GIÁ':<50} | " + " | ".join(f"{g:>11}" for g in GROUP_ORDER))
    print("-" * 130)
    for name, res in list(rainnet_audit_results.items()) + [(rn, rr) for rn, rr, _ in ref_entries]:
        jg = res.get("by_joint_group", {})
        for g in GROUP_ORDER:
            if g in jg:
                grp_rows_rn.append({"Protocol": name, "Joint_Group": g, **jg[g]})
        vals_str = " | ".join(f"{jg[g]['fMSE']:>11.2f}" if g in jg else f"{'N/A':>11}" for g in GROUP_ORDER)
        print(f"{name:<50} | {vals_str}")
    print("-" * 130)
    pd.DataFrame(grp_rows_rn).to_csv(RAINNET_AUDIT_DIR / "rainnet_audit_by_group.csv", index=False)

# ==============================================================================
# TRỰC QUAN HÓA: BIỂU ĐỒ BLIND vs OFFICIAL & LƯỚI ẢNH 4 CA NHẠY CẢM NHẤT
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
ds_keys = ARCHIVES + ["all"]
ds_lbls = ARCHIVES + ["ALL"]
x_pos = np.arange(len(ds_keys))
p_names = list(rainnet_audit_results.keys())
p_shorts = [p["short"].replace("\n", " ") for p in RAINNET_PROTOCOLS]
w_bar = 0.8 / len(p_names)
colors_rn = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]

for i, (pn, ps) in enumerate(zip(p_names, p_shorts)):
    off = (i - (len(p_names) - 1) / 2.0) * w_bar
    hatch = None if rainnet_audit_results[pn]["_meta"]["fair_blind"] else "//"
    axes[0].bar(
        x_pos + off, [rainnet_audit_results[pn][d]["fMSE"] for d in ds_keys],
        w_bar, label=ps, color=colors_rn[i], hatch=hatch, edgecolor="black", linewidth=0.6
    )
    axes[1].bar(
        x_pos + off, [rainnet_audit_results[pn][d]["PSNR"] for d in ds_keys],
        w_bar, label=ps, color=colors_rn[i], hatch=hatch, edgecolor="black", linewidth=0.6
    )

axes[0].set_title("RainNet 256: fMSE ↓ — Blind (đặc) vs Official GT-BG* (gạch chéo)", fontsize=11.5, fontweight="bold")
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(ds_lbls, fontweight="bold")
axes[0].set_ylabel("fMSE ↓"); axes[0].grid(axis="y", alpha=0.3)

axes[1].set_title("RainNet 256: PSNR ↑ — Blind (đặc) vs Official GT-BG* (gạch chéo)", fontsize=11.5, fontweight="bold")
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(ds_lbls, fontweight="bold")
axes[1].set_ylabel("PSNR (dB) ↑"); axes[1].set_ylim(bottom=29); axes[1].grid(axis="y", alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.12), fontsize=9.5)
plt.tight_layout()
chart_rn = RAINNET_AUDIT_DIR / "rainnet_blind_vs_official_chart.png"
plt.savefig(chart_rn, bbox_inches="tight", dpi=180)
plt.show()
print("Đã lưu biểu đồ:", chart_rn)

# Lưới ảnh trực quan: 1 ca nhạy cảm nhất mỗi sub-dataset (mask >= 1% để quan sát rõ)
df_r1 = rainnet_audit_dfs[RAINNET_PROTOCOLS[0]["name"]].set_index("full_id")
df_r3 = rainnet_audit_dfs[RAINNET_PROTOCOLS[2]["name"]].set_index("full_id")
row_by_fid = {r["full_id"]: r for r in records["test"]}
sensitive_rows = []
for sub in ARCHIVES:
    sub_fids = [fid for fid in df_r1.index if df_r1.loc[fid, "subset"] == sub]
    cand = [fid for fid in sub_fids if df_r1.loc[fid, "mask_ratio"] >= 0.01] or sub_fids
    best_fid = max(cand, key=lambda fid: df_r1.loc[fid, "fMSE"] - df_r3.loc[fid, "fMSE"])
    sensitive_rows.append(row_by_fid[best_fid])

ds_sens = HarmonyTestDataset(sensitive_rows, size=256)
cs, ms, ys = [], [], []
for i in range(len(ds_sens)):
    ci, mi, yi, _, _ = ds_sens[i]
    cs.append(ci); ms.append(mi); ys.append(yi)
c_b = torch.stack(cs).to(DEVICE)
m_b = torch.stack(ms).to(DEVICE)
y_b = torch.stack(ys).to(DEVICE)

out_r1, _ = infer_rainnet_mode(rn_model, c_b, m_b, y_b, False, False, False)
out_r2, _ = infer_rainnet_mode(rn_model, c_b, m_b, y_b, True, False, False)
out_r3, _ = infer_rainnet_mode(rn_model, c_b, m_b, y_b, True, True, False)

del rn_model
gc.collect()
torch.cuda.empty_cache()

cols_t = [c_b.cpu(), m_b.expand_as(c_b).cpu(), out_r1.cpu(), out_r2.cpu(), out_r3.cpu(), y_b.cpu()]
col_lbls = [
    "Input\nComposite", "Mask",
    "R1: Blind\n(KHÔNG gian lận)",
    "R2: GT-BG In\n+ Fair Comp BG",
    "R3: Official\nGT-BG* (rò rỉ)",
    "Ground\nTruth",
]
panels = torch.stack(cols_t, dim=1).flatten(0, 1)
PAD = 4
grid = make_grid(panels, nrow=len(col_lbls), padding=PAD, pad_value=1.0)
fig, ax = plt.subplots(figsize=(len(col_lbls) * 2.8, len(sensitive_rows) * 2.8 + 1.3))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis("off")
step = 256 + PAD
for idx, t in enumerate(col_lbls):
    ax.text(PAD + idx * step + 128, -10, t, ha="center", va="bottom", fontsize=10.5, fontweight="bold")
for r_idx, r in enumerate(sensitive_rows):
    fid = r["full_id"]
    d_blind = df_r1.loc[fid, "fMSE"]
    d_off = df_r3.loc[fid, "fMSE"]
    ax.text(
        -10, PAD + r_idx * step + 128,
        f"{r['subset']}\nBlind={d_blind:.0f}\nOfficial*={d_off:.0f}",
        ha="right", va="center", fontsize=9.5, fontweight="bold"
    )
fig.suptitle("Minh họa độ nhạy viền của RainNet 256: Blind (KHÔNG gian lận) vs Official GT-BG*", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
grid_rn = RAINNET_AUDIT_DIR / "rainnet_blind_vs_official_visual_grid.png"
plt.savefig(grid_rn, bbox_inches="tight", dpi=150)
plt.show()
print("Đã lưu lưới ảnh đối chiếu:", grid_rn)
print("Toàn bộ kết quả kiểm định riêng RainNet đã xuất vào:", RAINNET_AUDIT_DIR)


Sử dụng checkpoint RainNet gốc: /content/drive/MyDrive/ImageHarmonization/runs/RainNet_Pretrained/RainNet_net_G_last.pth



BẢNG R1: ĐÁNH GIÁ RIÊNG RAINNET (256x256) — KHÔNG GIAN LẬN (Blind) vs ĐÚNG CÁCH ĐO REPO GỐC (GT-BG*)
CẤU HÌNH ĐÁNH GIÁ                                  | Chế độ |  ms/im |   MSE ↓ |   fMSE ↓ |   ΔfMSE | PSNR ↑ | fPSNR ↑ | SSIM ↑ | fSSIM ↑ | fΔE76 ↓
------------------------------------------------------------------------------------------------------------------------------------------------
R1. RainNet — Blind (KHÔNG gian lận)               |  Blind |   6.87 |   58.09 |   619.18 |  +54.5% |  34.27 |   22.09 | 0.9806 |  0.8654 |   11.95
R2. RainNet — GT-BG In + Fair Comp BG              | GT-BG* |   8.30 |   48.74 |   531.44 |  +60.9% |  34.83 |   22.74 | 0.9816 |  0.8730 |   11.07
R3. RainNet — Official Repo (Float32, GT-BG*)      | GT-BG* |   6.97 |   46.76 |   531.44 |  +60.9% |  35.62 |   22.74 | 0.9868 |  0.8742 |   11.07
R4. RainNet — Official evaluate.py (uint8, GT-BG*) | GT-BG* |   7.04 |   47.27 |   536.63 |  +60.5% |  35.57 |   22.69 | 0.9865 |  0.8720 |   11.13
-------------

Đã lưu biểu đồ: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/rainnet_audit/rainnet_blind_vs_official_chart.png


Đã lưu lưới ảnh đối chiếu: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/rainnet_audit/rainnet_blind_vs_official_visual_grid.png
Toàn bộ kết quả kiểm định riêng RainNet đã xuất vào: /content/drive/MyDrive/ImageHarmonization/runs/evaluation_all_models/rainnet_audit
